# Deep Analysis — Per-Family, Severity, Confidence, Paired Subset

This notebook extends the Cross-MLLM Analysis with the analyses required to fully answer the three research questions:

- **RQ1 completion**: per-family detection rate per generator (the actual "artefact fingerprint") — Tables 6 and 7
- **RQ2 completion**: per-family Cohen's kappa and Krippendorff's alpha — Table 8
- **Severity distribution analysis**: L/M/H spread per family per MLLM — Table 9
- **Confidence analysis**: distribution and correctness relationship — Table 10
- **RQ3 completion**: paired 32-prompt subset analysis — Tables 11 and 12

All outputs (CSVs and PNGs) are written to the same `analysis_outputs/` folder used by the earlier notebook. Runtime ~1 minute.


## 1. Install and import

In [1]:
!pip install -q matplotlib numpy scipy
import json, pathlib, csv, math, re
from collections import Counter, defaultdict
from statistics import mean, stdev, median
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Configure paths

In [2]:
GEMINI_JSONL = "/content/drive/MyDrive/msc-deepfake/gemini_full_run/gemini_results.jsonl"
QWEN_JSONL   = "/content/drive/MyDrive/msc-deepfake/qwen_full_run/qwen_results.jsonl"

OUT_DIR = pathlib.Path("/content/drive/MyDrive/msc-deepfake/analysis_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

GENERATORS = ["LTX", "Hunyuan", "Wan", "Kling", "Gemini_Omni", "Seedance", "Pexels"]
AI_GENERATORS = ["LTX", "Hunyuan", "Wan", "Kling", "Gemini_Omni", "Seedance"]
FAMILIES = ["1.1_texture", "1.2_boundary", "1.3_lighting", "1.4_watermark",
            "2.1_human_anatomy", "2.2_non_human_anatomy", "2.3_object_structural",
            "2.4_scene_composition", "3.1_motion", "3.2_identity_drift",
            "3.3_continuity", "3.4_causality"]
FAMILY_LABELS = [f.split("_", 1)[1].replace("_", " ") for f in FAMILIES]
print(f"Loading from {GEMINI_JSONL}\n         and {QWEN_JSONL}")
print(f"Writing to  {OUT_DIR}")


Loading from /content/drive/MyDrive/msc-deepfake/gemini_full_run/gemini_results.jsonl
         and /content/drive/MyDrive/msc-deepfake/qwen_full_run/qwen_results.jsonl
Writing to  /content/drive/MyDrive/msc-deepfake/analysis_outputs


## 3. Load and dedupe records

In [3]:
def load_jsonl(path):
    recs = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: recs.append(json.loads(line))
            except json.JSONDecodeError: pass
    by_id = {}
    for r in recs:
        vid = r.get("video_id")
        if not vid: continue
        cur = by_id.get(vid)
        if cur is None:
            by_id[vid] = r; continue
        cur_ok = cur.get("status") == "ok"; new_ok = r.get("status") == "ok"
        if new_ok and not cur_ok:
            by_id[vid] = r
        elif new_ok == cur_ok and r.get("timestamp","") > cur.get("timestamp",""):
            by_id[vid] = r
    return list(by_id.values())

gem  = [r for r in load_jsonl(GEMINI_JSONL) if r.get("status") == "ok"]
qwen = [r for r in load_jsonl(QWEN_JSONL)   if r.get("status") == "ok"]
print(f"Gemini OK records: {len(gem)}")
print(f"Qwen OK records:   {len(qwen)}")

# Build lookup by video_id for pair analysis
gem_by_id  = {r["video_id"]: r for r in gem}
qwen_by_id = {r["video_id"]: r for r in qwen}
shared = sorted(set(gem_by_id) & set(qwen_by_id))
print(f"Shared videos (both MLLMs OK): {len(shared)}")


Gemini OK records: 279
Qwen OK records:   280
Shared videos (both MLLMs OK): 279


## 4. Per-family detection rate per generator (RQ1)

This is the artefact fingerprint promised in the methodology. For each MLLM, produce a 12×7 matrix showing what percentage of that generator's videos the MLLM detected each family in.


In [4]:
def build_fingerprint(records):
    # matrix[family][generator] = detection rate (0-1)
    counts = {f: {g: {"det": 0, "n": 0} for g in GENERATORS} for f in FAMILIES}
    for r in records:
        g = r.get("generator")
        if g not in GENERATORS: continue
        fams = (r.get("parsed") or {}).get("families", {})
        for f in FAMILIES:
            counts[f][g]["n"] += 1
            fam = fams.get(f, {})
            if fam.get("detected"):
                counts[f][g]["det"] += 1
    return counts

def print_fingerprint(name, counts):
    print(f"\n=== {name} — per-family detection rate per generator (%) ===")
    print(f"{'Family':22s}", end="")
    for g in GENERATORS: print(f"{g:>12s}", end="")
    print()
    for f in FAMILIES:
        print(f"{f:22s}", end="")
        for g in GENERATORS:
            c = counts[f][g]
            pct = (c["det"]/c["n"]*100) if c["n"] else 0
            print(f"{pct:>11.0f}%", end="")
        print()

gem_fp  = build_fingerprint(gem)
qwen_fp = build_fingerprint(qwen)
print_fingerprint("Gemini 3.1 Pro", gem_fp)
print_fingerprint("Qwen 3.5-397B-A17B", qwen_fp)

# Save
for name, fp in [("gemini", gem_fp), ("qwen", qwen_fp)]:
    path = OUT_DIR / f"table_family_by_generator_{name}.csv"
    with path.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["family"] + GENERATORS)
        for fam in FAMILIES:
            row = [fam]
            for g in GENERATORS:
                c = fp[fam][g]
                pct = (c["det"]/c["n"]*100) if c["n"] else 0
                row.append(f"{pct:.1f}")
            w.writerow(row)
    print(f"Wrote {path}")



=== Gemini 3.1 Pro — per-family detection rate per generator (%) ===
Family                         LTX     Hunyuan         Wan       Kling Gemini_Omni    Seedance      Pexels
1.1_texture                    92%         90%         82%         94%         97%         91%         59%
1.2_boundary                   92%         78%         70%         62%         69%         69%         51%
1.3_lighting                   45%         10%         22%         25%         28%         19%         17%
1.4_watermark                   2%          0%          2%         66%          3%          0%          2%
2.1_human_anatomy              75%         72%         70%         66%         75%         75%         57%
2.2_non_human_anatomy          18%         15%         15%         12%         12%          9%          3%
2.3_object_structural          78%         68%         65%         62%         72%         72%         51%
2.4_scene_composition          42%         20%         20%         19%    

## 5. Fingerprint heatmaps (Figures for Results)

In [5]:
def plot_fingerprint(counts, title, filename, cmap='Blues'):
    data = np.array([[(counts[f][g]["det"]/counts[f][g]["n"]*100 if counts[f][g]["n"] else 0)
                      for g in GENERATORS] for f in FAMILIES])
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(data, cmap=cmap, aspect='auto', vmin=0, vmax=100)
    ax.set_xticks(range(len(GENERATORS)))
    ax.set_xticklabels(["LTX", "Hunyuan", "Wan", "Kling", "Gemini\nOmni", "Seedance", "Pexels\n(Real)"], fontsize=10)
    ax.set_yticks(range(len(FAMILIES)))
    ax.set_yticklabels([f.replace("_", " ") for f in FAMILIES], fontsize=9)
    for i in range(len(FAMILIES)):
        for j in range(len(GENERATORS)):
            v = data[i, j]
            colour = 'white' if v > 55 else '#222'
            ax.text(j, i, f"{v:.0f}", ha='center', va='center', color=colour, fontsize=9, fontweight='bold')
    ax.set_xlabel("Generator (ground truth)", fontsize=11, fontweight='bold')
    ax.set_ylabel("Artefact family", fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    ax.set_xticks(np.arange(-.5, len(GENERATORS), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(FAMILIES), 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.4)
    ax.tick_params(which='minor', bottom=False, left=False)
    cbar = plt.colorbar(im, ax=ax, fraction=0.03); cbar.set_label('% detected', fontsize=10)
    plt.tight_layout(); plt.savefig(filename, dpi=150, bbox_inches='tight'); plt.close()
    print(f"Wrote {filename}")

plot_fingerprint(gem_fp,  "Gemini 3.1 Pro — artefact fingerprint",       OUT_DIR / "fig_fingerprint_gemini.png", 'Blues')
plot_fingerprint(qwen_fp, "Qwen 3.5-397B-A17B — artefact fingerprint",   OUT_DIR / "fig_fingerprint_qwen.png",   'Oranges')


Wrote /content/drive/MyDrive/msc-deepfake/analysis_outputs/fig_fingerprint_gemini.png
Wrote /content/drive/MyDrive/msc-deepfake/analysis_outputs/fig_fingerprint_qwen.png


## 6. Per-family Cohen's kappa and Krippendorff's alpha (RQ2)

Verdict-level kappa was 0.094 (slight). For each of the 12 families independently, compute the two agreement measures on the 279 shared videos. This shows whether the MLLMs disagree on the verdict only, or whether disagreement runs all the way down to individual family detections.


In [6]:
def cohen_kappa_binary(pairs):
    # pairs: list of (a, b) with a,b in {True, False}
    n = len(pairs)
    if n == 0: return None, None, None
    agree = sum(1 for a, b in pairs if a == b)
    po = agree / n
    a_pos = sum(1 for a, _ in pairs if a) / n
    b_pos = sum(1 for _, b in pairs if b) / n
    pe = a_pos*b_pos + (1-a_pos)*(1-b_pos)
    kappa = (po - pe) / (1 - pe) if pe < 1 else 0.0
    return kappa, po, pe

def krippendorff_alpha_nominal_binary(pairs):
    # For 2 coders, nominal data. Coincidence matrix approach.
    n_pairs = len(pairs)
    if n_pairs == 0: return None
    n_total = 2 * n_pairs
    row_sums = {True: 0, False: 0}
    for a, b in pairs:
        row_sums[a] += 1
        row_sums[b] += 1
    # Observed disagreement
    Do_num = sum(1 for a, b in pairs if a != b) * 2  # symmetric
    Do = Do_num / n_total
    # Expected disagreement
    De_num = 2 * row_sums[True] * row_sums[False]
    De_denom = n_total * (n_total - 1) if n_total > 1 else 1
    De = De_num / De_denom
    return 1 - Do / De if De else 0.0

# Compute for each family
print(f"{'Family':22s} {'n':>5s} {'Gem+%':>7s} {'Qwen+%':>7s} {'Kappa':>7s} {'Alpha':>7s} {'Agree%':>7s}")
per_family_agreement = []
for f in FAMILIES:
    pairs = []
    for vid in shared:
        g_fam = (gem_by_id[vid].get("parsed") or {}).get("families", {}).get(f, {})
        q_fam = (qwen_by_id[vid].get("parsed") or {}).get("families", {}).get(f, {})
        pairs.append((bool(g_fam.get("detected")), bool(q_fam.get("detected"))))
    kappa, po, pe = cohen_kappa_binary(pairs)
    alpha = krippendorff_alpha_nominal_binary(pairs)
    g_pos = sum(1 for a, _ in pairs if a) / len(pairs) * 100
    q_pos = sum(1 for _, b in pairs if b) / len(pairs) * 100
    print(f"{f:22s} {len(pairs):5d} {g_pos:6.1f}% {q_pos:6.1f}% {kappa:>7.3f} {alpha:>7.3f} {po*100:>6.1f}%")
    per_family_agreement.append((f, len(pairs), g_pos, q_pos, kappa, alpha, po*100))

# Save
path = OUT_DIR / "table_family_agreement.csv"
with path.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["family", "n", "gemini_positive_pct", "qwen_positive_pct",
                "cohen_kappa", "krippendorff_alpha_nominal", "raw_agreement_pct"])
    for row in per_family_agreement:
        w.writerow([row[0], row[1], f"{row[2]:.2f}", f"{row[3]:.2f}",
                    f"{row[4]:.4f}", f"{row[5]:.4f}", f"{row[6]:.2f}"])
print(f"\nWrote {path}")


Family                     n   Gem+%  Qwen+%   Kappa   Alpha  Agree%
1.1_texture              279   83.5%   29.4%   0.096  -0.172   42.3%
1.2_boundary             279   68.8%   29.0%   0.053  -0.095   45.2%
1.3_lighting             279   23.3%    5.4%   0.178   0.126   78.5%
1.4_watermark            279    9.0%    7.9%   0.930   0.930   98.9%
2.1_human_anatomy        279   68.8%   20.1%   0.158  -0.043   48.4%
2.2_non_human_anatomy    279   11.5%    4.7%   0.500   0.493   92.5%
2.3_object_structural    279   65.2%   22.2%   0.191   0.040   52.7%
2.4_scene_composition    279   22.6%    3.6%   0.168   0.103   79.6%
3.1_motion               279   69.9%   29.4%   0.008  -0.152   42.3%
3.2_identity_drift       279   68.1%    9.3%   0.025  -0.327   36.9%
3.3_continuity           279   39.1%    7.5%   0.084  -0.041   62.7%
3.4_causality            279   41.9%    7.9%   0.146   0.015   63.1%

Wrote /content/drive/MyDrive/msc-deepfake/analysis_outputs/table_family_agreement.csv


## 7. Severity distribution per family per MLLM

For each family, count L/M/H across all detections. Answers whether the M-collapse observed in the earlier note is family-specific or universal, and whether Qwen shows the same pattern as Gemini.


In [7]:
def severity_by_family(records):
    counts = {f: {"L": 0, "M": 0, "H": 0, "none": 0} for f in FAMILIES}
    for r in records:
        fams = (r.get("parsed") or {}).get("families", {})
        for f in FAMILIES:
            fam = fams.get(f, {})
            if fam.get("detected"):
                sev = fam.get("severity", "none")
                if sev in counts[f]:
                    counts[f][sev] += 1
    return counts

gem_sev  = severity_by_family(gem)
qwen_sev = severity_by_family(qwen)

def print_sev(name, sev):
    print(f"\n=== {name} — severity distribution (detections only) ===")
    print(f"{'Family':22s} {'L':>6s} {'M':>6s} {'H':>6s} {'total':>6s}   {'L%':>5s} {'M%':>5s} {'H%':>5s}")
    tot_L = tot_M = tot_H = 0
    for f in FAMILIES:
        c = sev[f]
        total = c["L"] + c["M"] + c["H"]
        Lp = (c["L"]/total*100) if total else 0
        Mp = (c["M"]/total*100) if total else 0
        Hp = (c["H"]/total*100) if total else 0
        print(f"{f:22s} {c['L']:6d} {c['M']:6d} {c['H']:6d} {total:6d}   {Lp:4.0f}% {Mp:4.0f}% {Hp:4.0f}%")
        tot_L += c["L"]; tot_M += c["M"]; tot_H += c["H"]
    grand = tot_L + tot_M + tot_H
    if grand:
        print(f"{'OVERALL':22s} {tot_L:6d} {tot_M:6d} {tot_H:6d} {grand:6d}   "
              f"{tot_L/grand*100:4.0f}% {tot_M/grand*100:4.0f}% {tot_H/grand*100:4.0f}%")

print_sev("Gemini 3.1 Pro", gem_sev)
print_sev("Qwen 3.5-397B-A17B", qwen_sev)

# Save
path = OUT_DIR / "table_severity_by_family.csv"
with path.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["family", "mllm", "L", "M", "H", "total"])
    for name, sev in [("gemini", gem_sev), ("qwen", qwen_sev)]:
        for fam in FAMILIES:
            c = sev[fam]
            w.writerow([fam, name, c["L"], c["M"], c["H"], c["L"]+c["M"]+c["H"]])
print(f"\nWrote {path}")



=== Gemini 3.1 Pro — severity distribution (detections only) ===
Family                      L      M      H  total      L%    M%    H%
1.1_texture                14    219      0    233      6%   94%    0%
1.2_boundary               22    170      0    192     11%   89%    0%
1.3_lighting                6     59      0     65      9%   91%    0%
1.4_watermark               0     25      0     25      0%  100%    0%
2.1_human_anatomy           1    191      0    192      1%   99%    0%
2.2_non_human_anatomy       0     32      0     32      0%  100%    0%
2.3_object_structural       4    178      0    182      2%   98%    0%
2.4_scene_composition       0     63      0     63      0%  100%    0%
3.1_motion                  6    189      0    195      3%   97%    0%
3.2_identity_drift          6    184      0    190      3%   97%    0%
3.3_continuity              5    104      0    109      5%   95%    0%
3.4_causality               1    116      0    117      1%   99%    0%
OVERALL    

## 8. Confidence analysis

Distribution of the confidence integer (0–100) per MLLM, plus relationship between confidence and correctness at the verdict level.


In [8]:
def confidence_stats(records, name):
    confs = []
    correct = []   # 1 if verdict matches source ground truth
    for r in records:
        p = r.get("parsed") or {}
        c = p.get("confidence")
        if not isinstance(c, (int, float)): continue
        confs.append(c)
        # Correctness
        gt_ai = r.get("generator") != "Pexels"
        pred = p.get("video_verdict")
        pred_ai = pred == "AI-generated"
        # Correctness only defined when the MLLM commits to a class
        if pred in ("AI-generated", "Real"):
            correct.append(1 if pred_ai == gt_ai else 0)
    print(f"\n=== {name} — confidence ===")
    print(f"  n = {len(confs)}")
    print(f"  mean   = {mean(confs):.1f}")
    print(f"  sd     = {stdev(confs):.1f}")
    print(f"  median = {median(confs):.1f}")
    print(f"  min    = {min(confs)}")
    print(f"  max    = {max(confs)}")
    # Distribution in 10-buckets
    buckets = [0]*10
    for c in confs:
        b = min(int(c/10), 9)
        buckets[b] += 1
    print(f"  distribution (10-bucket): {buckets}")
    # Correctness by confidence band
    print(f"  correctness by confidence band:")
    bands = [(0,49), (50,69), (70,79), (80,100)]
    for lo, hi in bands:
        idxs = [i for i, c in enumerate(confs) if lo <= c <= hi]
        n_band = len(idxs)
        n_correct = sum(correct[i] for i in idxs if i < len(correct))
        if n_band:
            print(f"    {lo:>3d}–{hi:<3d}: n={n_band:3d}, correct={n_correct}/{len(idxs)} = {n_correct/len(idxs)*100:.0f}%")
    return confs, correct

gem_confs, gem_corr = confidence_stats(gem, "Gemini 3.1 Pro")
qwen_confs, qwen_corr = confidence_stats(qwen, "Qwen 3.5-397B-A17B")

# Save
path = OUT_DIR / "table_confidence.csv"
with path.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["mllm", "n", "mean", "sd", "median", "min", "max"])
    for name, cs in [("gemini", gem_confs), ("qwen", qwen_confs)]:
        if cs:
            w.writerow([name, len(cs), f"{mean(cs):.2f}", f"{stdev(cs):.2f}",
                        f"{median(cs):.1f}", min(cs), max(cs)])
print(f"\nWrote {path}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, cs, colour) in zip(axes, [("Gemini 3.1 Pro", gem_confs, "#2E4A7B"),
                                          ("Qwen 3.5-397B-A17B", qwen_confs, "#B85C1F")]):
    ax.hist(cs, bins=range(0, 105, 5), color=colour, edgecolor='white', alpha=0.85)
    ax.set_xlim(0, 100)
    ax.set_xlabel("Confidence"); ax.set_ylabel("Count")
    ax.set_title(f"{name} (n={len(cs)}, mean={mean(cs):.1f}, sd={stdev(cs):.1f})")
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_confidence_dist.png", dpi=150, bbox_inches='tight'); plt.close()
print(f"Wrote {OUT_DIR / 'fig_confidence_dist.png'}")



=== Gemini 3.1 Pro — confidence ===
  n = 279
  mean   = 66.0
  sd     = 13.4
  median = 70.0
  min    = 25
  max    = 75
  distribution (10-bucket): [0, 0, 3, 31, 3, 3, 11, 228, 0, 0]
  correctness by confidence band:
      0–49 : n= 37, correct=17/37 = 46%
     50–69 : n= 14, correct=13/14 = 93%
     70–79 : n=228, correct=196/228 = 86%

=== Qwen 3.5-397B-A17B — confidence ===
  n = 280
  mean   = 90.4
  sd     = 9.6
  median = 92.0
  min    = 15
  max    = 100
  distribution (10-bucket): [0, 3, 0, 0, 2, 0, 0, 3, 45, 227]
  correctness by confidence band:
      0–49 : n=  5, correct=2/5 = 40%
     70–79 : n=  3, correct=1/3 = 33%
     80–100: n=272, correct=141/272 = 52%

Wrote /content/drive/MyDrive/msc-deepfake/analysis_outputs/table_confidence.csv
Wrote /content/drive/MyDrive/msc-deepfake/analysis_outputs/fig_confidence_dist.png


## 9. Paired 32-prompt subset analysis (RQ3)

The methodology commits to answering RQ3 on the 32-prompt subset where all six AI generators produced videos from the same prompt. Extract that subset from the video IDs (prompt id = the `w2_XXX` prefix), and compute per-MLLM metrics restricted to it.


In [9]:
# Extract prompt id from video_id like 'w2_005_ltx_20260719_105736'
PROMPT_RE = re.compile(r"^(w2_\d{3})_")

def prompt_id(vid):
    m = PROMPT_RE.match(vid or "")
    return m.group(1) if m else None

# For each AI generator, count which prompts produced videos
prompt_by_gen = defaultdict(set)   # generator -> set of prompt ids
all_ai_records = [r for r in gem if r.get("generator") in AI_GENERATORS]
for r in all_ai_records:
    pid = prompt_id(r["video_id"])
    if pid:
        prompt_by_gen[r["generator"]].add(pid)

# The paired subset: prompts where every AI generator produced a video
paired_prompts = set(prompt_by_gen[AI_GENERATORS[0]])
for g in AI_GENERATORS[1:]:
    paired_prompts &= prompt_by_gen[g]
paired_prompts = sorted(paired_prompts)
print(f"Paired-subset prompts (all 6 generators produced a video): {len(paired_prompts)}")
print(f"First few: {paired_prompts[:5]}  Last few: {paired_prompts[-5:]}")

# Restrict both MLLM record sets to the paired subset
def paired_records(records):
    return [r for r in records if r.get("generator") in AI_GENERATORS
            and prompt_id(r["video_id"]) in set(paired_prompts)]

gem_p  = paired_records(gem)
qwen_p = paired_records(qwen)
print(f"Gemini records on paired subset: {len(gem_p)}  (expected 6 x {len(paired_prompts)} = {6*len(paired_prompts)})")
print(f"Qwen records on paired subset:   {len(qwen_p)}")

def detection_rate_per_gen(records):
    out = {}
    for g in AI_GENERATORS:
        vids = [r for r in records if r.get("generator") == g]
        n = len(vids)
        detected = sum(1 for r in vids if (r.get("parsed") or {}).get("video_verdict") == "AI-generated")
        out[g] = (detected, n, detected/n if n else 0)
    return out

gem_p_rate  = detection_rate_per_gen(gem_p)
qwen_p_rate = detection_rate_per_gen(qwen_p)

print(f"\n=== Detection rate on paired 32-prompt subset ===")
print(f"{'Generator':14s}  {'Gemini':>18s}  {'Qwen':>18s}")
for g in AI_GENERATORS:
    gd, gn, gp = gem_p_rate[g]
    qd, qn, qp = qwen_p_rate[g]
    print(f"{g:14s}  {gd:3d}/{gn:2d} = {gp*100:5.1f}%   {qd:3d}/{qn:2d} = {qp*100:5.1f}%")

# Save paired-subset table
path = OUT_DIR / "table_paired_subset_detection.csv"
with path.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["generator", "n_paired", "gemini_detected", "gemini_rate", "qwen_detected", "qwen_rate"])
    for g in AI_GENERATORS:
        gd, gn, gp = gem_p_rate[g]
        qd, qn, qp = qwen_p_rate[g]
        w.writerow([g, gn, gd, f"{gp:.4f}", qd, f"{qp:.4f}"])
print(f"\nWrote {path}")

# --- Per-prompt difficulty on the paired subset ---
# For each prompt, count how many of the 12 verdicts (6 gens x 2 MLLMs) said 'AI-generated'
per_prompt = []
for pid in paired_prompts:
    votes = 0
    total = 0
    for g in AI_GENERATORS:
        for r in gem_p + qwen_p:
            if r.get("generator") == g and prompt_id(r["video_id"]) == pid:
                total += 1
                if (r.get("parsed") or {}).get("video_verdict") == "AI-generated":
                    votes += 1
    per_prompt.append((pid, votes, total))
per_prompt.sort(key=lambda x: x[1])

print(f"\n=== Hardest 5 prompts (fewest AI verdicts across 12 detectors) ===")
for pid, v, t in per_prompt[:5]:
    print(f"  {pid}: {v}/{t} AI verdicts across 6 generators × 2 MLLMs")
print(f"\n=== Easiest 5 prompts (most AI verdicts) ===")
for pid, v, t in per_prompt[-5:]:
    print(f"  {pid}: {v}/{t}")

path = OUT_DIR / "table_paired_per_prompt_difficulty.csv"
with path.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["prompt_id", "ai_verdicts", "total_verdicts", "detection_rate"])
    for pid, v, t in per_prompt:
        w.writerow([pid, v, t, f"{v/t:.4f}" if t else "0"])
print(f"\nWrote {path}")


Paired-subset prompts (all 6 generators produced a video): 32
First few: ['w2_001', 'w2_002', 'w2_003', 'w2_004', 'w2_006']  Last few: ['w2_034', 'w2_036', 'w2_037', 'w2_038', 'w2_039']
Gemini records on paired subset: 192  (expected 6 x 32 = 192)
Qwen records on paired subset:   192

=== Detection rate on paired 32-prompt subset ===
Generator                   Gemini                Qwen
LTX              29/32 =  90.6%     9/32 =  28.1%
Hunyuan          29/32 =  90.6%     8/32 =  25.0%
Wan              28/32 =  87.5%     6/32 =  18.8%
Kling            31/32 =  96.9%    24/32 =  75.0%
Gemini_Omni      32/32 = 100.0%    15/32 =  46.9%
Seedance         30/32 =  93.8%    13/32 =  40.6%

Wrote /content/drive/MyDrive/msc-deepfake/analysis_outputs/table_paired_subset_detection.csv

=== Hardest 5 prompts (fewest AI verdicts across 12 detectors) ===
  w2_028: 2/12 AI verdicts across 6 generators × 2 MLLMs
  w2_004: 6/12 AI verdicts across 6 generators × 2 MLLMs
  w2_018: 6/12 AI verdicts across

## 10. Summary of outputs

Everything written to `analysis_outputs/`:


In [10]:
outputs = sorted(OUT_DIR.glob("*"))
print(f"{'File':50s}  {'Size':>8s}")
for p in outputs:
    if p.is_file():
        sz = p.stat().st_size
        sz_str = f"{sz/1024:.1f} KB" if sz < 1024*1024 else f"{sz/1024/1024:.1f} MB"
        print(f"{p.name:50s}  {sz_str:>8s}")

print(f"\nAll deep-analysis outputs ready. Insert the CSVs into Results as Tables 6–12,")
print(f"and the two new PNG figures alongside the earlier confusion-matrix heatmaps.")


File                                                    Size
fig_cm_gemini.png                                    89.4 KB
fig_cm_qwen.png                                      95.9 KB
fig_confidence_dist.png                              49.0 KB
fig_cross_mllm.png                                   43.4 KB
fig_fingerprint_gemini.png                          126.2 KB
fig_fingerprint_qwen.png                            118.9 KB
table_binary_metrics.csv                              0.2 KB
table_confidence.csv                                  0.1 KB
table_cross_mllm.csv                                  0.3 KB
table_difficulty_per_generator.csv                    0.2 KB
table_family_agreement.csv                            0.7 KB
table_family_by_generator_gemini.csv                  0.7 KB
table_family_by_generator_qwen.csv                    0.6 KB
table_paired_per_prompt_difficulty.csv                0.7 KB
table_paired_subset_detection.csv                     0.2 KB
table_severity_by_family